In [0]:
# Employees table
emp_data = [
    (1,  "Ravi",   1,  55000),
    (2,  "Priya",  2,  42000),
    (3,  "Arjun",  1,  72000),
    (4,  "Sneha",  3,  61000),
    (5,  "Rohit",  1,  80000),
    (6,  "Meera",  2,  39000),
    (7,  "Karan",  3,  55000),
    (8,  "Divya",  1,  91000),
    (9,  "Nitin",  99, 44000),   # dept_id 99 = no match
    (10, "Anjali", None, 67000), # null dept_id
]
emp_cols = ["emp_id", "name", "dept_id", "salary"]
emp_df = spark.createDataFrame(emp_data, emp_cols)

# Departments table
dept_data = [
    (1, "Engineering", "Pune"),
    (2, "HR",          "Mumbai"),
    (3, "Finance",     "Delhi"),
    (4, "Marketing",   "Bangalore"),  # no employees in this dept
]
dept_cols = ["dept_id", "dept_name", "location"]
dept_df = spark.createDataFrame(dept_data, dept_cols)

# Salary grades table
grade_data = [
    (1, "Low",    0,     50000),
    (2, "Medium", 50001, 70000),
    (3, "High",   70001, 999999),
]
grade_cols = ["grade_id", "grade", "min_sal", "max_sal"]
grade_df = spark.createDataFrame(grade_data, grade_cols)

display(emp_df)
display(dept_df)

emp_id,name,dept_id,salary
1,Ravi,1,55000
2,Priya,2,42000
3,Arjun,1,72000
4,Sneha,3,61000
5,Rohit,1,80000
6,Meera,2,39000
7,Karan,3,55000
8,Divya,1,91000
9,Nitin,99,44000
10,Anjali,null,67000


dept_id,dept_name,location
1,Engineering,Pune
2,HR,Mumbai
3,Finance,Delhi
4,Marketing,Bangalore


In [0]:
result = emp_df.join(dept_df, on="dept_id", how="inner")
display(result)

dept_id,emp_id,name,salary,dept_name,location
1,1,Ravi,55000,Engineering,Pune
2,2,Priya,42000,HR,Mumbai
1,3,Arjun,72000,Engineering,Pune
3,4,Sneha,61000,Finance,Delhi
1,5,Rohit,80000,Engineering,Pune
2,6,Meera,39000,HR,Mumbai
3,7,Karan,55000,Finance,Delhi
1,8,Divya,91000,Engineering,Pune


In [0]:
result2 = emp_df.join(dept_df, on="dept_id", how="inner") \
    .select("name", "salary", "dept_name", "location")
display(result2)

# Notice: Nitin (dept_id=99) and Anjali (null) are GONE
# Notice: Marketing dept (id=4) is GONE

name,salary,dept_name,location
Ravi,55000,Engineering,Pune
Priya,42000,HR,Mumbai
Arjun,72000,Engineering,Pune
Sneha,61000,Finance,Delhi
Rohit,80000,Engineering,Pune
Meera,39000,HR,Mumbai
Karan,55000,Finance,Delhi
Divya,91000,Engineering,Pune


In [0]:
# Left join — ALL employees + matching dept info
# If no dept match → dept columns are null
result = emp_df.join(dept_df, on="dept_id", how="left")
display(result)
# Notice: Nitin (dept_id=99) still here but dept_name = null
# Notice: Marketing dept (id=4) is NOT here

dept_id,emp_id,name,salary,dept_name,location
1,1,Ravi,55000,Engineering,Pune
2,2,Priya,42000,HR,Mumbai
1,3,Arjun,72000,Engineering,Pune
3,4,Sneha,61000,Finance,Delhi
1,5,Rohit,80000,Engineering,Pune
2,6,Meera,39000,HR,Mumbai
3,7,Karan,55000,Finance,Delhi
1,8,Divya,91000,Engineering,Pune
99,9,Nitin,44000,null,null
null,10,Anjali,67000,null,null


In [0]:
# Practical use — find employees with no department

from pyspark.sql.functions import *
no_dept=emp_df.join(dept_df, on = "dept_id", how="left").filter(col("dept_name").isNull())
display(no_dept)

dept_id,emp_id,name,salary,dept_name,location
99,9,Nitin,44000,null,null
null,10,Anjali,67000,null,null


In [0]:
# Right join — ALL departments + matching employees
result = emp_df.join(dept_df, on="dept_id", how="right")
display(result)
# Notice: Marketing dept appears with null employee data
# Notice: Nitin (dept_id=99) is GONE

dept_id,emp_id,name,salary,dept_name,location
1,8,Divya,91000,Engineering,Pune
1,5,Rohit,80000,Engineering,Pune
1,3,Arjun,72000,Engineering,Pune
1,1,Ravi,55000,Engineering,Pune
2,6,Meera,39000,HR,Mumbai
2,2,Priya,42000,HR,Mumbai
3,7,Karan,55000,Finance,Delhi
3,4,Sneha,61000,Finance,Delhi
4,null,null,null,Marketing,Bangalore


In [0]:
# Full join — EVERYTHING from both sides
result = emp_df.join(dept_df, on="dept_id", how="full")
display(result)
# Notice: Nitin here (no dept match → null dept cols)
# Notice: Marketing here (no emp match → null emp cols)

# Also called: "full", "outer", "full_outer" — all same
result2 = emp_df.join(dept_df, on="dept_id", how="full_outer")

dept_id,emp_id,name,salary,dept_name,location
1,1,Ravi,55000,Engineering,Pune
2,2,Priya,42000,HR,Mumbai
1,3,Arjun,72000,Engineering,Pune
3,4,Sneha,61000,Finance,Delhi
1,5,Rohit,80000,Engineering,Pune
2,6,Meera,39000,HR,Mumbai
3,7,Karan,55000,Finance,Delhi
1,8,Divya,91000,Engineering,Pune
99,9,Nitin,44000,null,null
null,10,Anjali,67000,null,null


In [0]:
# left_anti — employees with NO matching department
# Most used for finding orphan records / data quality checks
no_dept = emp_df.join(dept_df, on="dept_id", how="left_anti")
display(no_dept)
# Returns: Nitin (99) and Anjali (null)

dept_id,emp_id,name,salary
99,9,Nitin,44000
null,10,Anjali,67000


In [0]:
# left_semi — employees who DO have a matching department
# Like inner join but returns ONLY left table columns
has_dept = emp_df.join(dept_df, on="dept_id", how="left_semi")
display(has_dept)
# Returns emp columns only — no dept columns added

dept_id,emp_id,name,salary
1,1,Ravi,55000
2,2,Priya,42000
1,3,Arjun,72000
3,4,Sneha,61000
1,5,Rohit,80000
2,6,Meera,39000
3,7,Karan,55000
1,8,Divya,91000


In [0]:
# Sometimes you need to match on more than one column
# Create sample data
sales1 = spark.createDataFrame([
    ("Ravi", "2024-01", 5000),
    ("Priya","2024-01", 3000),
    ("Ravi", "2024-02", 7000),
], ["name", "month", "sales"])

target = spark.createDataFrame([
    ("Ravi", "2024-01", 4500),
    ("Priya","2024-01", 3500),
    ("Ravi", "2024-02", 6000),
], ["name", "month", "target"])

# Join on name AND month
result = sales1.join(target, on=["name","month"], how="inner")
display(result)

name,month,sales,target
Ravi,2024-01,5000,4500
Priya,2024-01,3000,3500
Ravi,2024-02,7000,6000


In [0]:
from pyspark.sql.functions import broadcast

# dept_df is small (4 rows) — broadcast it to all executors
# Avoids shuffle — much faster for large emp_df
result = emp_df.join(broadcast(dept_df), on="dept_id", how="inner")
display(result)

# Rule: broadcast when one table < 10MB
# Databricks auto-broadcasts if table < threshold
# You can set threshold:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

dept_id,emp_id,name,salary,dept_name,location
1,1,Ravi,55000,Engineering,Pune
2,2,Priya,42000,HR,Mumbai
1,3,Arjun,72000,Engineering,Pune
3,4,Sneha,61000,Finance,Delhi
1,5,Rohit,80000,Engineering,Pune
2,6,Meera,39000,HR,Mumbai
3,7,Karan,55000,Finance,Delhi
1,8,Divya,91000,Engineering,Pune


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7524220066182069>, line 11
      6 display(result)
      8 # Rule: broadcast when one table < 10MB
      9 # Databricks auto-broadcasts if table < threshold
     10 # You can set threshold:
---> 11 spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:51, in RuntimeConf.set(self, key, value)
     49 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     50 operation = proto.ConfigRequest.Operation(set=op_set)
---> 51 result = self._client.config(operation)
     52 for warn in result.warnings:
     53     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2193, in SparkConnectClient.config(self, operation)
   2191     raise SparkConnectExc

In [0]:
# Problem: both tables have "dept_id" → ambiguous after join
emp_df2 = emp_df.withColumnRenamed("dept_id", "emp_dept_id")

# Method 1 — rename before join
result = emp_df2.join(dept_df,
    emp_df2.emp_dept_id == dept_df.dept_id,
    how="inner")
display(result)

# Method 2 — alias the DataFrames
result2 = emp_df.alias("e").join(
    dept_df.alias("d"),
    col("e.dept_id") == col("d.dept_id"),
    how="inner"
).select("e.name", "e.salary", "d.dept_name", "d.location")
display(result2)

# Method 3 — drop duplicate column after join
result3 = emp_df.join(dept_df, on="dept_id", how="inner") \
    .drop(dept_df.dept_id)
display(result3)

emp_id,name,emp_dept_id,salary,dept_id,dept_name,location
1,Ravi,1,55000,1,Engineering,Pune
2,Priya,2,42000,2,HR,Mumbai
3,Arjun,1,72000,1,Engineering,Pune
4,Sneha,3,61000,3,Finance,Delhi
5,Rohit,1,80000,1,Engineering,Pune
6,Meera,2,39000,2,HR,Mumbai
7,Karan,3,55000,3,Finance,Delhi
8,Divya,1,91000,1,Engineering,Pune


name,salary,dept_name,location
Ravi,55000,Engineering,Pune
Priya,42000,HR,Mumbai
Arjun,72000,Engineering,Pune
Sneha,61000,Finance,Delhi
Rohit,80000,Engineering,Pune
Meera,39000,HR,Mumbai
Karan,55000,Finance,Delhi
Divya,91000,Engineering,Pune


dept_id,emp_id,name,salary,dept_name,location
1,1,Ravi,55000,Engineering,Pune
2,2,Priya,42000,HR,Mumbai
1,3,Arjun,72000,Engineering,Pune
3,4,Sneha,61000,Finance,Delhi
1,5,Rohit,80000,Engineering,Pune
2,6,Meera,39000,HR,Mumbai
3,7,Karan,55000,Finance,Delhi
1,8,Divya,91000,Engineering,Pune
